# Test: `RegressionPicker` on California Housing

This notebook imports `regression_picker_1a.py` directly and uses the built-in `plot_` methods for the charts. Section 10 calls every chart method and checks each one draws without errors.

**Dataset:** California Housing (1990 census, one row per block group), as used in Aurélien Géron's *Hands-On Machine Learning*.
About 20,000 rows. The target is `median_house_value`.

It's a good test because it has:
- a **text column** (`ocean_proximity`), which checks the automatic one-hot encoding
- **missing values** in `total_bedrooms`, which shows the "clean df first" rule
- **non-linear patterns** (location, income), so tree and boosting models should beat linear ones

The notebook ends with automatic checks that print **All checks passed** if the module behaves as expected.

In [ ]:
# --- Find the ML root folder and import the picker ---
import sys
from pathlib import Path

ml_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pickers").is_dir())
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

%load_ext autoreload
%autoreload 2

from pickers.regression_picker_1a import RegressionPicker
print("Using:", ml_root / "pickers" / "regression_picker_1a.py")
import matplotlib.pyplot as plt

## 1. Load the data

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
raw = pd.read_csv(URL)
print(raw.shape)
raw.head()

## 2. Clean it
The picker expects a **clean** DataFrame and will refuse one with missing values. Here we just drop the ~200 incomplete rows.

Note: house values were capped at \$500,001 in the census data, so the very top end is harder to predict for every model.

In [ ]:
print("Missing values before:\n", raw.isna().sum()[raw.isna().sum() > 0])

df = raw.dropna().reset_index(drop=True)
print("\nRows after cleaning:", len(df))
df["ocean_proximity"].value_counts()

## 3. Optional: use a sample for a faster run
With `SAMPLE_N = 5000` the full notebook takes a minute or two. Set it to `None` to use all ~20,000 rows.
With the full data, SVR and Quantile Regression are skipped automatically because they are too slow at that size.

In [ ]:
SAMPLE_N = 5000          # None = use every row

data = df.sample(SAMPLE_N, random_state=42) if SAMPLE_N else df
print("Rows used:", len(data))

## 4. Set up the picker and look at the starting hyperparameters

In [ ]:
rp = RegressionPicker(data, target="median_house_value", sort="R2")

print(rp.summary())
rp.models()

In [ ]:
# Starting values are worked out from the data, not the library defaults
for m in ["lasso", "knn", "dt", "hgb"]:
    print(f"{m:6}", rp.starting_params(m))

## 5. Compare all models (5-fold CV on the training split)

In [ ]:
%%time
leaderboard = rp.compare_models()
leaderboard

In [ ]:
rp.plot_leaderboard();

In [ ]:
# Any metric, top models only (the dummy baseline is always kept)
rp.plot_leaderboard(metric="RMSE", top=8);

## 6. Tune the top 3
Each model is tuned with a random search. The tuned version is kept only if it beats the starting version in cross-validation.
Then both are checked on the 20% holdout set that hasn't been touched yet.

In [ ]:
%%time
tuned = rp.tune_top(n=3, n_iter=20)
tuned

In [ ]:
rp.plot_tuning();

In [ ]:
print(rp.summary())

In [ ]:
# Recommended hyperparameters for every tuned model
for model_id, params in rp.recommended_params_.items():
    print(model_id, params)

## 7. Charts for the best model (holdout set)
Each `plot_` method uses the best model unless you pass a model id, e.g. `rp.plot_residuals("rf")`.

In [ ]:
rp.plot_prediction_error();

In [ ]:
rp.plot_residuals();

Feature importance is measured on the **original columns** (`ocean_proximity` is one bar, not five one-hot pieces): how much worse the model gets when that column is shuffled.

In [ ]:
rp.plot_feature_importance();

## 8. Use the final model
`finalize()` refits the winner on **all** rows (train + holdout), ready for new data.

In [ ]:
final_model = rp.finalize()

new_rows = data.drop(columns="median_house_value").head(5)
pd.DataFrame({"actual": data["median_house_value"].head(5).values,
              "predicted": final_model.predict(new_rows).round(0)})

## 9. Automatic checks

In [ ]:
import numpy as np

lb, tu = rp.leaderboard_, rp.tuned_
best = tu.index[0]

checks = {
    "leaderboard has 15+ models":                   len(lb) >= 15,
    "dummy baseline is in the leaderboard":         "dummy" in lb.index,
    "best CV R2 beats dummy by 0.3+":               lb["R2"].max() > lb.loc["dummy", "R2"] + 0.3,
    "text column was one-hot encoded":              rp.cat_cols == ["ocean_proximity"],
    "3 models were tuned":                          len(tu) == 3,
    "tuned models have recommended params":         all(m in rp.recommended_params_ for m in tu.index),
    "best holdout R2 is above 0.70":                tu.loc[best, "Holdout R2"] > 0.70,
    "a tree/boosting model won (data is non-linear)": rp.specs[best].family == "ensemble",
    "final model predicts one value per row":       len(final_model.predict(new_rows)) == len(new_rows),
    "missing values are rejected":                  False,
}

try:
    RegressionPicker(raw, target="median_house_value")
except ValueError:
    checks["missing values are rejected"] = True

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

assert all(checks.values()), "Some checks failed - see above"
print("\nAll checks passed")

## 10. Chart checks
Calls every `plot_` method, including non-default options. Each must return a matplotlib chart without an error.

In [ ]:
from matplotlib.axes import Axes
from matplotlib.figure import Figure

def run_chart_checks(calls, expect_errors=None):
    """Call every chart; PASS if it returns a matplotlib Axes/Figure without an error.
    expect_errors: {name: (callable, ExceptionType)} that SHOULD raise that error."""
    results = {}
    for name, call in calls.items():
        try:
            results[name] = isinstance(call(), (Axes, Figure))
        except Exception as e:
            print(f"  {name} -> {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, (call, err) in (expect_errors or {}).items():
        try:
            call()
            results[name] = False
        except err:
            results[name] = True
        except Exception as e:
            print(f"  {name} -> unexpected {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, ok in results.items():
        print(("PASS  " if ok else "FAIL  ") + name)
    return results


In [ ]:
second = rp.tuned_.index[1]          # a model other than the winner

calls = {
    "plot_leaderboard()":                       lambda: rp.plot_leaderboard(),
    "plot_leaderboard(metric='MAE', top=5)":    lambda: rp.plot_leaderboard(metric="MAE", top=5),
    "plot_tuning()":                            lambda: rp.plot_tuning(),
    "plot_prediction_error()":                  lambda: rp.plot_prediction_error(),
    "plot_prediction_error(second model)":      lambda: rp.plot_prediction_error(second),
    "plot_residuals()":                         lambda: rp.plot_residuals(),
    "plot_feature_importance(top=5)":           lambda: rp.plot_feature_importance(top=5, n_repeats=2),
    "plot_ draws onto an ax you supply":        lambda: rp.plot_leaderboard(ax=plt.subplots()[1]),
}
expect = {
    "bad metric name raises ValueError":        (lambda: rp.plot_leaderboard(metric="Accuracy"), ValueError),
    "plot_tuning() before tune_top() raises":   (lambda: RegressionPicker(data, target="median_house_value").plot_tuning(), RuntimeError),
}
chart_results = run_chart_checks(calls, expect)
assert all(chart_results.values()), "Some chart checks failed - see above"
print("\nAll chart checks passed")